# Imports

In [1]:
import os
import pandas as pd
import numpy as np


# Configurações para exibição de DataFrames
pd.set_option('display.max_columns', None)

# Funções úteis

### padronizar_colunas_ano

In [2]:
def padronizar_colunas_ano(df: pd.DataFrame, ano: int, ignorar_cols: list = None) -> pd.DataFrame:
    """
    Padroniza os nomes das colunas para snake_case, adiciona o sufixo do ano
    e converte para maiúsculas no final do processo.
    Retorna uma cópia do DataFrame, mantendo o original inalterado.

    Args:
        df (pd.DataFrame): O DataFrame original.
        ano (int): O ano de referência.
        ignorar_cols (list, optional): Lista de colunas a ignorar.

    Returns:
        pd.DataFrame: Novo DataFrame com colunas padronizadas.
    """
    # Cria uma cópia independente para não afetar o df original
    df_out = df.copy()
    
    if ignorar_cols is None:
        ignorar_cols = []
    
    ano_str = str(ano)        
    ano_curto = ano_str[-2:]  
    
    novas_colunas = []
    
    for col in df_out.columns:
        if col in ignorar_cols:
            novas_colunas.append(col.upper())
            continue
            
        # 1. Ajuste inicial: remove espaços e substitui por underline (mantém case original)
        col_nova = col.strip().replace(" ", "_")
        
        # 2. Adiciona sufixo se necessário
        if not (col_nova.endswith(ano_str) or col_nova.endswith(ano_curto)):
            col_nova = f"{col_nova}_{ano_curto}"
        
        # 3. Converte para maiúsculas apenas no final
        col_nova = col_nova.upper()
            
        novas_colunas.append(col_nova)
    
    df_out.columns = novas_colunas
    return df_out

### adicionar_colunas_vazias

In [3]:
def adicionar_colunas_vazias(df: pd.DataFrame, novas_colunas: list) -> pd.DataFrame:
    """
    Adiciona colunas preenchidas com NaN ao DataFrame caso elas não existam.
    Retorna uma cópia do DataFrame, mantendo o original inalterado.

    Args:
        df (pd.DataFrame): O DataFrame original.
        novas_colunas (list): Lista de strings com os nomes das colunas a adicionar.

    Returns:
        pd.DataFrame: Novo DataFrame com as colunas adicionadas.
    """
    # Cria uma cópia para não alterar o original
    df_out = df.copy()
    
    for col in novas_colunas:
        if col not in df_out.columns:
            df_out[col] = np.nan
            
    return df_out

### analise_nulos

In [4]:
def analise_nulos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um relatório contendo a quantidade e a porcentagem de valores nulos por coluna.

    Args:
        df (pd.DataFrame): O DataFrame a ser analisado.

    Returns:
        pd.DataFrame: Um DataFrame contendo as colunas 'nome_coluna', 'qtd_nulos' 
                      e 'porcentagem_nulos', ordenado decrescentemente pela quantidade de nulos.
    """
    # Cálculo das métricas
    nulos = df.isnull().sum()
    porcentagem = ((nulos / len(df)) * 100).round(2)
    
    # Criação do DataFrame de resumo
    df_nulos = pd.DataFrame({
        'qtd_nulos': nulos,
        'porcentagem_nulos': porcentagem
    })
    
    # Ajuste do índice para se tornar uma coluna
    df_nulos.reset_index(inplace=True)
    df_nulos.rename(columns={'index': 'nome_coluna'}, inplace=True)
    
    # Ordenação
    df_nulos.sort_values(by='qtd_nulos', ascending=False, inplace=True)
    
    return df_nulos

### calcular_idade_2023

In [5]:
def calcular_idade_2023(df: pd.DataFrame, col_data_nasc: str, col_nova_idade: str) -> pd.DataFrame:
    """
    Calcula a idade dos alunos referenciada ao ano de 2023 a partir de uma data de nascimento.
    
    A função converte a coluna de datas para o formato datetime (esperando 'mês/dia/ano')
    e subtrai o ano de nascimento de 2023.
    Retorna uma cópia do DataFrame com a nova coluna de idade.

    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        col_data_nasc (str): O nome da coluna que contém as datas de nascimento (formato 'mm/dd/yyyy').
        col_nova_idade (str): O nome da nova coluna que será criada com a idade calculada.

    Returns:
        pd.DataFrame: Um novo DataFrame com a coluna de idade adicionada.
    """
    df_out = df.copy()
    
    # Converte para datetime forçando o formato mês/dia/ano e lidando com erros
    datas_convertidas = pd.to_datetime(df_out[col_data_nasc], format='%m/%d/%Y', errors='coerce')
    
    # Calcula a idade (2023 - ano de nascimento)
    # Fillna(-1) ou manter NaN pode ser opção, aqui mantemos NaN para datas inválidas
    df_out[col_nova_idade] = 2023 - datas_convertidas.dt.year
    
    return df_out

### obter_elementos_comuns

In [6]:
def obter_elementos_comuns(df1, df2, coluna):
    """
    Retorna um set com os elementos presentes em ambos os dataframes.
    Extremamente rápido para grandes volumes de dados.
    """
    set1 = set(df1[coluna])
    set2 = set(df2[coluna])
    
    # A operação & realiza a interseção entre os conjuntos
    return set1 & set2

### obter_nova_turma

In [7]:
def obter_nova_turma(df: pd.DataFrame, coluna_turma: str) -> pd.DataFrame:
    """
    Cria uma nova coluna com a identificação da turma padronizada.
    
    A lógica aplicada é:
    1. Se houver espaço, pega o segundo elemento (ex: '8 A' -> 'A').
    2. Se não houver espaço, remove o primeiro caractere (ex: '8A' -> 'A').
    3. Valores nulos ou inválidos são retornados como NaN.

    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        coluna_turma (str): O nome da coluna original da turma.

    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a coluna 'NOVA_{coluna_turma}' adicionada.
        
    Raises:
        ValueError: Se a coluna especificada não existir no DataFrame.
    """
    if coluna_turma not in df.columns:
        raise ValueError(f"Coluna '{coluna_turma}' não encontrada no DataFrame.")
    
    df_out = df.copy()
    
    def extrair_sufixo(valor):
        # Retorna NaN se o valor for nulo
        if pd.isna(valor):
            return np.nan
            
        val_str = str(valor).strip()
        
        if ' ' in val_str:
            partes = val_str.split()
            # Garante que existe o segundo elemento após o split
            return partes[1] if len(partes) > 1 else np.nan
        else:
            # Retorna o slice apenas se a string tiver tamanho suficiente
            return val_str[1:] if len(val_str) > 1 else val_str

    # Aplica a lógica linha a linha de forma segura
    nova_coluna = f"NOVA_{coluna_turma}"
    df_out[nova_coluna] = df_out[coluna_turma].apply(extrair_sufixo)
    
    return df_out

### obter_nova_fase

In [8]:
def obter_nova_fase(df: pd.DataFrame, coluna_fase_ideal: str) -> pd.DataFrame:
    """
    Cria uma nova coluna com a identificação da fase padronizada.
    
    A lógica aplicada é:
    1. Se contiver 'ALFA', retorna 'Fase 0'.
    2. Se contiver espaço (ex: 'Fase 1 (4º ano)'), retorna os dois primeiros elementos ('Fase 1').
    3. Caso contrário, retorna o valor original.

    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        coluna_fase_ideal (str): O nome da coluna original da fase ideal.

    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a coluna 'NOVA_{coluna_fase_ideal}' adicionada.
        
    Raises:
        ValueError: Se a coluna especificada não existir no DataFrame.
    """
    if coluna_fase_ideal not in df.columns:
        raise ValueError(f"Coluna '{coluna_fase_ideal}' não encontrada no DataFrame.")
    
    df_out = df.copy()
    
    def extrair_sufixo(valor):
        if pd.isna(valor):
            return np.nan
            
        val_str = str(valor).strip()
        
        # Verifica ALFA (independente de maiúsculas/minúsculas para segurança)
        if 'ALFA' in val_str.upper():
            return 'Fase 0'
        
        # Lógica para 'Fase X ...'
        partes = val_str.split()
        if len(partes) >= 2:
            return f"{partes[0]} {partes[1]}"
            
        return val_str

    # Define o nome da nova coluna
    nova_coluna = f"NOVA_{coluna_fase_ideal}"
    df_out[nova_coluna] = df_out[coluna_fase_ideal].apply(extrair_sufixo)
    
    return df_out

### obter_nova_fase_24

In [9]:
def obter_nova_fase_24(df: pd.DataFrame, col_fase: str) -> pd.DataFrame:
    """
    Cria uma nova coluna padronizada para as fases de 2024.
    
    Regras aplicadas:
    1. Se contiver 'ALFA', retorna 'Fase 0'.
    2. Se o valor for 9 (inteiro ou string), retorna 'Fase 8'.
    3. Para códigos alfanuméricos (ex: '4F', '3N'), extrai o primeiro dígito 
       e formata como 'Fase X'.
    
    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        col_fase (str): O nome da coluna original (ex: 'FASE_24').
        
    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a coluna 'NOVA_{col_fase}' adicionada.
        
    Raises:
        ValueError: Se a coluna especificada não existir no DataFrame.
    """
    if col_fase not in df.columns:
        raise ValueError(f"Coluna '{col_fase}' não encontrada no DataFrame.")
    
    df_out = df.copy()
    
    def padronizar_elemento(valor):
        if pd.isna(valor):
            return np.nan
        
        # Converte para string, remove espaços e coloca em maiúsculo
        val_str = str(valor).strip().upper()
        
        # Regra 1: ALFA
        if 'ALFA' in val_str:
            return 'Fase 0'
        
        # Regra 2: Caso específico do número 9
        if val_str == '9':
            return 'Fase 8'
        
        # Regra 3: Extração do primeiro dígito (ex: '4F' -> 'Fase 4')
        # Verifica se o primeiro caractere é numérico
        if val_str and val_str[0].isdigit():
            return f"Fase {val_str[0]}"
            
        # Retorno padrão caso não se encaixe nas regras (segurança)
        return val_str

    nova_coluna = f"NOVA_{col_fase}"
    df_out[nova_coluna] = df_out[col_fase].apply(padronizar_elemento)
    
    return df_out

### obter_nova_turma_24

In [10]:
def obter_nova_turma_24(df: pd.DataFrame, col_turma: str) -> pd.DataFrame:
    """
    Cria uma nova coluna com a letra identificadora da turma de 2024.
    
    Regras aplicadas:
    1. Se for 9 ou '9', retorna NaN.
    2. Se contiver 'ALFA' (ex: 'ALFA M - G0/G1'), extrai a letra logo após a palavra ALFA (ex: 'M').
    3. Para códigos alfanuméricos (ex: '5B', '8E'), extrai o último caractere (ex: 'B', 'E').
    
    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        col_turma (str): O nome da coluna original da turma (ex: 'TURMA_24').
        
    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a coluna 'NOVA_{col_turma}' adicionada.
        
    Raises:
        ValueError: Se a coluna especificada não existir no DataFrame.
    """
    if col_turma not in df.columns:
        raise ValueError(f"Coluna '{col_turma}' não encontrada no DataFrame.")
    
    df_out = df.copy()
    
    def extrair_letra(valor):
        if pd.isna(valor):
            return np.nan
        
        # Converte para string, remove espaços e coloca em maiúsculo
        val_str = str(valor).strip().upper()
        
        # Regra 1: Valor 9 é nulo
        if val_str == '9':
            return np.nan
        
        # Regra 2: Turmas ALFA (ex: "ALFA M - ...")
        if 'ALFA' in val_str:
            partes = val_str.split()
            # Retorna o segundo elemento (índice 1) se existir
            return partes[1] if len(partes) > 1 else np.nan
            
        # Regra 3: Turmas padrão (ex: "5B") -> Pega o último caractere
        if len(val_str) > 0:
            return val_str[-1]
            
        return np.nan

    nova_coluna = f"NOVA_{col_turma}"
    df_out[nova_coluna] = df_out[col_turma].apply(extrair_letra)
    
    return df_out

### criar_coluna_veterano

In [11]:
def criar_coluna_veterano(df: pd.DataFrame, col_ano_ingresso: str, ano_referencia: int) -> pd.DataFrame:
    """
    Cria uma coluna binária 'VETERANO' baseada no ano de ingresso do aluno.
    
    A lógica aplicada é:
    1. Se o ano de ingresso for menor que o ano de referência, o aluno é veterano (retorna 1).
    2. Se o ano de ingresso for igual ao ano de referência, não é veterano (retorna 0).
    3. Valores nulos ou inválidos são retornados como NaN.

    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        col_ano_ingresso (str): O nome da coluna que contém o ano de ingresso.
        ano_referencia (int): O ano utilizado como base para a verificação.

    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a coluna 'VETERANO' adicionada.
        
    Raises:
        ValueError: Se a coluna especificada não existir no DataFrame.
    """
    if col_ano_ingresso not in df.columns:
        raise ValueError(f"Coluna '{col_ano_ingresso}' não encontrada no DataFrame.")
    
    df_out = df.copy()
    
    def classificar_veterano(valor):
        if pd.isna(valor):
            return np.nan
        
        try:
            # Garante a comparação numérica
            ano = int(valor)
            return 1 if ano < ano_referencia else 0
        except ValueError:
            # Retorna NaN caso o valor não possa ser convertido para inteiro
            return np.nan

    df_out['VETERANO'] = df_out[col_ano_ingresso].apply(classificar_veterano)
    
    return df_out

### criar_coluna_em_fase

In [12]:
def criar_coluna_em_fase(df: pd.DataFrame, col_fase_atual: str, col_fase_ideal: str) -> pd.DataFrame:
    """
    Cria uma coluna binária 'EM_FASE' indicando se o aluno está na fase ideal.
    
    A lógica aplicada é:
    1. Se a fase atual for igual à fase ideal, retorna 1.
    2. Se a fase atual for diferente da fase ideal, retorna 0.
    3. Se houver valores nulos em qualquer uma das colunas comparadas, retorna NaN.

    Args:
        df (pd.DataFrame): O DataFrame contendo os dados.
        col_fase_atual (str): O nome da coluna que contém a fase atual do aluno.
        col_fase_ideal (str): O nome da coluna que contém a fase ideal do aluno.

    Returns:
        pd.DataFrame: Uma cópia do DataFrame com a nova coluna 'EM_FASE'.
        
    Raises:
        ValueError: Se alguma das colunas especificadas não existir no DataFrame.
    """
    if col_fase_atual not in df.columns or col_fase_ideal not in df.columns:
        raise ValueError(f"Colunas '{col_fase_atual}' e/ou '{col_fase_ideal}' não encontradas no DataFrame.")
    
    df_out = df.copy()
    
    # Identificação de nulos para evitar falsos negativos na comparação
    nulos = df_out[col_fase_atual].isna() | df_out[col_fase_ideal].isna()
    
    # Comparação de igualdade
    iguais = df_out[col_fase_atual] == df_out[col_fase_ideal]
    
    # Aplicação das condições (nulos recebem NaN, iguais recebem 1, diferentes recebem 0)
    df_out['EM_FASE'] = np.where(nulos, np.nan, np.where(iguais, 1, 0))
    
    return df_out

# Preparando os dados

In [ ]:
data_dir = '../app/data/raw'
file_name = 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'

file_path = os.path.join(data_dir, file_name)

sheets = ['PEDE2022', 'PEDE2023', 'PEDE2024']

dataframes = {}

## Carregando e definindo dados

In [14]:
if os.path.exists(file_path):
    try:
        excel_file = pd.ExcelFile(file_path)
        
        for sheet in sheets:
            if sheet in excel_file.sheet_names:
                print(f"Processando aba: {sheet}...")
                
                # Leitura da aba
                df = pd.read_excel(excel_file, sheet_name=sheet)
                
                # Definição do caminho de saída para o CSV (salvando na mesma pasta 'data')
                csv_filename = os.path.join(data_dir, f"{sheet}.csv")
                
                # Salvamento em CSV
                df.to_csv(csv_filename, index=False, encoding='utf-8')
                print(f"  -> Salvo em: {csv_filename}")
                
                # Armazenamento no dicionário
                dataframes[sheet] = df
            else:
                print(f"Aviso: Aba '{sheet}' não encontrada.")
        
        # Criação das variáveis de DataFrame solicitadas
        df_pede_2022 = dataframes.get('PEDE2022')
        df_pede_2023 = dataframes.get('PEDE2023')
        df_pede_2024 = dataframes.get('PEDE2024')
        
        print("\nProcessamento concluído.")
        
        # Conferência das dimensões
        if df_pede_2022 is not None: print(f"df_pede_2022: {df_pede_2022.shape}")
        if df_pede_2023 is not None: print(f"df_pede_2023: {df_pede_2023.shape}")
        if df_pede_2024 is not None: print(f"df_pede_2024: {df_pede_2024.shape}")

    except Exception as e:
        print(f"Erro ao processar o arquivo: {e}")
else:
    print(f"Arquivo não encontrado no caminho: {os.path.abspath(file_path)}")
    print("Verifique se o arquivo .xlsx foi movido para a pasta 'data'.")

Processando aba: PEDE2022...
  -> Salvo em: ../data/raw/PEDE2022.csv
Processando aba: PEDE2023...
  -> Salvo em: ../data/raw/PEDE2023.csv
Processando aba: PEDE2024...
  -> Salvo em: ../data/raw/PEDE2024.csv

Processamento concluído.
df_pede_2022: (860, 42)
df_pede_2023: (1014, 48)
df_pede_2024: (1156, 50)


In [15]:
# Criação de conjuntos (sets) com os nomes das colunas
cols_2022 = set(df_pede_2022.columns)
cols_2023 = set(df_pede_2023.columns)
cols_2024 = set(df_pede_2024.columns)

# Identificação de colunas comuns e exclusivas
comuns = cols_2022 & cols_2023 & cols_2024
print(f"Colunas comuns aos 3 anos ({len(comuns)}): \n{sorted(list(comuns))}\n")

# Diferença de conjuntos (Colunas do ano - Colunas comuns)
excedentes_2022 = cols_2022 - comuns
excedentes_2023 = cols_2023 - comuns
excedentes_2024 = cols_2024 - comuns

print(f"Excedentes 2022 ({len(excedentes_2022)}): \n{sorted(list(excedentes_2022))}\n")
print(f"Excedentes 2023 ({len(excedentes_2023)}): \n{sorted(list(excedentes_2023))}\n")
print(f"Excedentes 2024 ({len(excedentes_2024)}): \n{sorted(list(excedentes_2024))}\n")

Colunas comuns aos 3 anos (32): 
['Ano ingresso', 'Atingiu PV', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Cf', 'Cg', 'Ct', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Fase', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE 22', 'IPS', 'IPV', 'Indicado', 'Instituição de ensino', 'Nº Av', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'RA', 'Rec Av1', 'Rec Av2', 'Rec Psicologia', 'Turma']

Excedentes 2022 (10): 
['Ano nasc', 'Defas', 'Fase ideal', 'Idade 22', 'Inglês', 'Matem', 'Nome', 'Portug', 'Rec Av3', 'Rec Av4']

Excedentes 2023 (16): 
['Data de Nasc', 'Defasagem', 'Destaque IPV.1', 'Fase Ideal', 'INDE 2023', 'INDE 23', 'IPP', 'Idade', 'Ing', 'Mat', 'Nome Anonimizado', 'Pedra 2023', 'Pedra 23', 'Por', 'Rec Av3', 'Rec Av4']

Excedentes 2024 (18): 
['Ativo/ Inativo', 'Ativo/ Inativo.1', 'Avaliador5', 'Avaliador6', 'Data de Nasc', 'Defasagem', 'Escola', 'Fase Ideal', 'INDE 2024', 'INDE 23', 'IPP', 'Idade', 'Ing', 'Mat', 'Nome Anonimizado', 'Pedra 2024', 'Pedra 23', 'Por']



In [16]:
df_pede_2022 = df_pede_2022.drop(columns=[col for col in df_pede_2022.columns if col.endswith('.1')], errors='ignore')
df_pede_2023 = df_pede_2023.drop(columns=[col for col in df_pede_2023.columns if col.endswith('.1')], errors='ignore')
df_pede_2024 = df_pede_2024.drop(columns=[col for col in df_pede_2024.columns if col.endswith('.1')], errors='ignore')

columns_2_drop_2023 = ['INDE 23', 'Pedra 23']
df_pede_2023 = df_pede_2023.drop(columns=columns_2_drop_2023, errors='ignore')

# Adicionando colunas que faltam aos dataframes
df_pede_2024 = adicionar_colunas_vazias(df_pede_2024, ['Rec Av3', 'Rec Av4', 'Rec Av5', 'Rec Av6'])

In [17]:
# Dicionário de padronização 2022
mapa_padronizacao_2022 = {
    'Ano ingresso': 'Ano_Ingresso',
    'Ano nasc': 'Data_de_Nasc',
    'Defas': 'Defasagem',
    'Fase ideal': 'Fase_Ideal',
    'Instituição de ensino': 'Instituicao_Ensino',
    'Idade 22': 'Idade_2022',
    'Inglês': 'Ingles',
    'Matem': 'Matematica',
    'Nome': 'Nome_Anonimizado',
    'Nº Av': 'QTDE_AVAL_2022',
    'Portug': 'Portugues',
    'INDE 22' : 'INDE_2022',
}
# Aplicando a renomeação no DataFrame de 2022
df_pede_2022.rename(columns=mapa_padronizacao_2022, inplace=True)

# Dicionário de padronização 2023
mapa_padronizacao_2023 = {
    'Ano ingresso': 'Ano_Ingresso',
    'Data de Nasc': 'Data_de_Nasc',
    'Fase Ideal': 'Fase_Ideal',
    'Instituição de ensino': 'Instituicao_Ensino',
    'Idade': 'Idade_2023',
    'Ing': 'Ingles',
    'Mat': 'Matematica',
    'Nome Anonimizado': 'Nome_Anonimizado',
    'Nº Av': 'QTDE_AVAL_2023',
    'Por': 'Portugues',
    'IPP': 'IPP_2023',
    'INDE 22': 'INDE_2022'
}
# Aplicando a renomeação no DataFrame de 2023
df_pede_2023.rename(columns=mapa_padronizacao_2023, inplace=True)

# Dicionário de padronização 2024
mapa_padronizacao_2024 = {
    'Ano ingresso': 'Ano_Ingresso',
    'Data de Nasc': 'Data_de_Nasc',
    'Fase Ideal': 'Fase_Ideal',
    'Instituição de ensino': 'Instituicao_Ensino',
    'Idade': 'Idade_2024',
    'Ing': 'Ingles',
    'Mat': 'Matematica',
    'Nome Anonimizado': 'Nome_Anonimizado',
    'Nº Av': 'QTDE_AVAL_2024',
    'Por': 'Portugues',
    'IPP': 'IPP_2024',
    'INDE 22': 'INDE_2022',
    'INDE 23': 'INDE_2023',
    'INDE 2024': 'INDE_2024',
    'Pedra 2024': 'Pedra 24'

}
# Aplicando a renomeação no DataFrame de 2024
df_pede_2024.rename(columns=mapa_padronizacao_2024, inplace=True)

In [18]:
# Criação de conjuntos (sets) com os nomes das colunas
cols_2022 = set(df_pede_2022.columns)
cols_2023 = set(df_pede_2023.columns)
cols_2024 = set(df_pede_2024.columns)

# Identificação de colunas comuns e exclusivas
comuns = cols_2022 & cols_2023 & cols_2024
print(f"Colunas comuns aos 3 anos ({len(comuns)}): \n{sorted(list(comuns))}\n")

# Diferença de conjuntos (Colunas do ano - Colunas comuns)
excedentes_2022 = cols_2022 - comuns
excedentes_2023 = cols_2023 - comuns
excedentes_2024 = cols_2024 - comuns

print(f"Excedentes 2022 ({len(excedentes_2022)}): \n{sorted(list(excedentes_2022))}\n")
print(f"Excedentes 2023 ({len(excedentes_2023)}): \n{sorted(list(excedentes_2023))}\n")
print(f"Excedentes 2024 ({len(excedentes_2024)}): \n{sorted(list(excedentes_2024))}\n")

Colunas comuns aos 3 anos (40): 
['Ano_Ingresso', 'Atingiu PV', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Cf', 'Cg', 'Ct', 'Data_de_Nasc', 'Defasagem', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Fase', 'Fase_Ideal', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE_2022', 'IPS', 'IPV', 'Indicado', 'Ingles', 'Instituicao_Ensino', 'Matematica', 'Nome_Anonimizado', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Portugues', 'RA', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Turma']

Excedentes 2022 (2): 
['Idade_2022', 'QTDE_AVAL_2022']

Excedentes 2023 (5): 
['INDE 2023', 'IPP_2023', 'Idade_2023', 'Pedra 2023', 'QTDE_AVAL_2023']

Excedentes 2024 (13): 
['Ativo/ Inativo', 'Avaliador5', 'Avaliador6', 'Escola', 'INDE_2023', 'INDE_2024', 'IPP_2024', 'Idade_2024', 'Pedra 23', 'Pedra 24', 'QTDE_AVAL_2024', 'Rec Av5', 'Rec Av6']



## Padronizando dataframes

### PEDE 2022

In [19]:
df_pede_2022_padronizado = padronizar_colunas_ano(df_pede_2022, 2022, ignorar_cols=['Ano_Ingresso', 'Data_de_Nasc', 'Gênero', 'Nome_Anonimizado', 'RA', 'Pedra 20', 'Pedra 21'])
display(df_pede_2022_padronizado.head())

,RA,FASE_22,TURMA_22,NOME_ANONIMIZADO,DATA_DE_NASC,IDADE_2022,GÊNERO,ANO_INGRESSO,INSTITUICAO_ENSINO_22,PEDRA 20,PEDRA 21,PEDRA_22,INDE_2022,CG_22,CF_22,CT_22,QTDE_AVAL_2022,AVALIADOR1_22,REC_AV1_22,AVALIADOR2_22,REC_AV2_22,AVALIADOR3_22,REC_AV3_22,AVALIADOR4_22,REC_AV4_22,IAA_22,IEG_22,IPS_22,REC_PSICOLOGIA_22,IDA_22,MATEMATICA_22,PORTUGUES_22,INGLES_22,INDICADO_22,ATINGIU_PV_22,IPV_22,IAN_22,FASE_IDEAL_22,DEFASAGEM_22,DESTAQUE_IEG_22,DESTAQUE_IDA_22,DESTAQUE_IPV_22
0,RA-1,7,A,Aluno-1,2003,19,Menina,2016,Escola Pública,Ametista,Ametista,Quartzo,5.783,753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,8.3,4.1,5.6,Requer avaliação,4.0,2.7,3.5,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,7,A,Aluno-2,2005,17,Menina,2017,Rede Decisão,Ametista,Ametista,Ametista,7.055,469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,8.8,5.2,6.3,Sem limitações,6.8,6.3,4.5,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,7,A,Aluno-3,2005,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ágata,6.591,629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,0.0,7.9,5.6,Sem limitações,5.6,5.8,4.0,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
3,RA-4,7,A,Aluno-4,2005,17,Menino,2017,Rede Decisão,Ametista,Ametista,Quartzo,5.951,731,15,7,4,Avaliador-5,Promovido de Fase,Avaliador-27,Mantido na Fase atual,Avaliador-28,Mantido na Fase atual,Avaliador-31,Mantido na Fase atual,8.8,4.5,5.6,Requer avaliação,5.0,2.8,3.5,8.7,Não,Não,5.278,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
4,RA-5,7,A,Aluno-5,2005,17,Menina,2016,Rede Decisão,Ametista,Ametista,Ametista,7.427,344,6,2,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,7.9,8.6,5.6,Requer avaliação,5.2,7.0,2.9,5.7,Não,Não,7.389,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...


In [20]:
mapa_genero = {'Menina': 1, 'Menino': 0}
df_pede_2022_padronizado['GÊNERO'] = df_pede_2022_padronizado['GÊNERO'].map(mapa_genero)

mapa_pv = {'Sim': 1, 'Não': 0}
df_pede_2022_padronizado['ATINGIU_PV_22'] = df_pede_2022_padronizado['ATINGIU_PV_22'].map(mapa_pv)

# Verificando os valores únicos da coluna de fase
mapa_fase_22 = {
    0: 'Fase 0',
    1: 'Fase 1',
    2: 'Fase 2',
    3: 'Fase 3',
    4: 'Fase 4',
    5: 'Fase 5',
    6: 'Fase 6',
    7: 'Fase 7',
}
df_pede_2022_padronizado['FASE_22'] = df_pede_2022_padronizado['FASE_22'].map(mapa_fase_22)

df_pede_2022_padronizado = criar_coluna_veterano(df_pede_2022_padronizado, 'ANO_INGRESSO', 2022)

# Verificando os valores únicos da coluna de instituição de ensino
mapa_instituicao_ensino_22 = {
    'Rede Decisão': 'Privada',
    'Escola Pública': 'Pública',
    'Escola JP II': 'Privada',
}
df_pede_2022_padronizado['INSTITUICAO_ENSINO_22'] = df_pede_2022_padronizado['INSTITUICAO_ENSINO_22'].map(mapa_instituicao_ensino_22)

# Obtendo a nova coluna de fase ideal padronizada
df_pede_2022_padronizado = obter_nova_fase(df_pede_2022_padronizado, 'FASE_IDEAL_22')

df_pede_2022_padronizado = criar_coluna_em_fase(df_pede_2022_padronizado, 'FASE_22', 'NOVA_FASE_IDEAL_22') 

In [21]:
df_pede_2022_padronizado.info()

<class 'pandas.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 45 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     860 non-null    str    
 1   FASE_22                860 non-null    str    
 2   TURMA_22               860 non-null    str    
 3   NOME_ANONIMIZADO       860 non-null    str    
 4   DATA_DE_NASC           860 non-null    int64  
 5   IDADE_2022             860 non-null    int64  
 6   GÊNERO                 860 non-null    int64  
 7   ANO_INGRESSO           860 non-null    int64  
 8   INSTITUICAO_ENSINO_22  860 non-null    str    
 9   PEDRA 20               323 non-null    str    
 10  PEDRA 21               462 non-null    str    
 11  PEDRA_22               860 non-null    str    
 12  INDE_2022              860 non-null    float64
 13  CG_22                  860 non-null    int64  
 14  CF_22                  860 non-null    int64  
 15  CT_22            

### PEDE 2023

In [22]:
df_pede_2023_padronizado = padronizar_colunas_ano(df_pede_2023, 2023, ignorar_cols=['Ano_Ingresso', 'Data_de_Nasc', 'Gênero', 'INDE_2022','Nome_Anonimizado', 'RA', 'Pedra 20', 'Pedra 21', 'Pedra 22'])
display(df_pede_2023_padronizado.head())

,RA,FASE_23,INDE_2023,PEDRA_2023,TURMA_23,NOME_ANONIMIZADO,DATA_DE_NASC,IDADE_2023,GÊNERO,ANO_INGRESSO,INSTITUICAO_ENSINO_23,PEDRA 20,PEDRA 21,PEDRA 22,INDE_2022,CG_23,CF_23,CT_23,QTDE_AVAL_2023,AVALIADOR1_23,REC_AV1_23,AVALIADOR2_23,REC_AV2_23,AVALIADOR3_23,REC_AV3_23,AVALIADOR4_23,REC_AV4_23,IAA_23,IEG_23,IPS_23,IPP_2023,REC_PSICOLOGIA_23,IDA_23,MATEMATICA_23,PORTUGUES_23,INGLES_23,INDICADO_23,ATINGIU_PV_23,IPV_23,IAN_23,FASE_IDEAL_23,DEFASAGEM_23,DESTAQUE_IEG_23,DESTAQUE_IDA_23,DESTAQUE_IPV_23
0,RA-861,ALFA,9.31095,Topázio,ALFA A - G0/G1,Aluno-861,6/17/2015,8,Feminino,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Avaliador-11,NaN,Avaliador-2,NaN,NaN,NaN,NaN,NaN,9.5,10.0,8.13,8.4375,NaN,9.6,9.8,9.4,NaN,NaN,NaN,8.920,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN
1,RA-862,ALFA,8.22120,Topázio,ALFA A - G0/G1,Aluno-862,5/31/2014,9,Masculino,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Avaliador-11,NaN,Avaliador-2,NaN,NaN,NaN,NaN,NaN,8.5,9.1,8.14,7.5000,NaN,8.9,8.5,9.2,NaN,NaN,NaN,8.585,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN
2,RA-863,ALFA,5.92975,Quartzo,ALFA A - G0/G1,Aluno-863,2/25/2016,7,Masculino,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Avaliador-11,NaN,Avaliador-2,NaN,NaN,NaN,NaN,NaN,0.0,7.6,3.14,5.9375,NaN,6.3,7.0,5.5,NaN,NaN,NaN,6.260,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN
3,RA-864,ALFA,7.03400,Ametista,ALFA A - G0/G1,Aluno-864,2015-12-03 00:00:00,1900-01-08 00:00:00,Feminino,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Avaliador-11,NaN,Avaliador-2,NaN,NaN,NaN,NaN,NaN,0.0,7.6,8.14,7.5000,NaN,6.3,7.0,5.5,NaN,NaN,NaN,8.500,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN
4,RA-865,ALFA,8.15520,Topázio,ALFA A - G0/G1,Aluno-865,11/13/2014,8,Masculino,2023,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,Avaliador-11,NaN,Avaliador-2,NaN,NaN,NaN,NaN,NaN,8.5,8.7,7.52,7.5000,NaN,7.4,7.3,7.5,NaN,NaN,NaN,7.915,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN


In [23]:
mapa_genero = {'Feminino': 1, 'Masculino': 0}
df_pede_2023_padronizado['GÊNERO'] = df_pede_2023_padronizado['GÊNERO'].map(mapa_genero)

mapa_fase_23 = {
    'ALFA': 'Fase 0',
    'FASE 1': 'Fase 1',
    'FASE 2': 'Fase 2',
    'FASE 3': 'Fase 3',
    'FASE 4': 'Fase 4',
    'FASE 5': 'Fase 5',
    'FASE 6': 'Fase 6',
    'FASE 7': 'Fase 7',
    'FASE 8': 'Fase 8',
}
df_pede_2023_padronizado['FASE_23'] = df_pede_2023_padronizado['FASE_23'].map(mapa_fase_23)

# Ajustando a idade de 2023 para ser calculada a partir da data de nascimento. Motivo: coluna original está errada
df_pede_2023_padronizado = calcular_idade_2023(df_pede_2023_padronizado, 'DATA_DE_NASC', 'NOVA_IDADE_2023')

# Criando nova coluna da turma para o ano de 2023
df_pede_2023_padronizado = obter_nova_turma(df_pede_2023_padronizado, 'TURMA_23')

# Obtendo a nova coluna de fase ideal padronizada
df_pede_2023_padronizado = obter_nova_fase(df_pede_2023_padronizado, 'FASE_IDEAL_23')

df_pede_2023_padronizado = criar_coluna_em_fase(df_pede_2023_padronizado, 'FASE_23', 'NOVA_FASE_IDEAL_23') 

df_pede_2023_padronizado = criar_coluna_veterano(df_pede_2023_padronizado, 'ANO_INGRESSO', 2023)

# Ajuste específico para corrigir erro de digitação na coluna de pedra de 2023
ajuste_pedra_23 = {
    'Agata': 'Ágata',
}
df_pede_2023_padronizado['PEDRA_2023'] = df_pede_2023_padronizado['PEDRA_2023'].replace(ajuste_pedra_23) 

mapa_instituicao_ensino_23 = {
    'Pública': 'Pública',
    'Privada': 'Privada',
    'Concluiu o 3º EM': 'Concluiu o 3º EM',
    'Privada - Programa de Apadrinhamento': 'Privada',
    'Privada *Parcerias com Bolsa 100%': 'Privada',
    'Privada - Pagamento por *Empresa Parceira': 'Privada',
    'Privada - Programa de apadrinhamento': 'Privada',
    'Nenhuma das opções acima': 'Nenhuma das opções acima'
}
df_pede_2023_padronizado['INSTITUICAO_ENSINO_23'] = df_pede_2023_padronizado['INSTITUICAO_ENSINO_23'].map(mapa_instituicao_ensino_23)

In [24]:
df_pede_2023_padronizado.info()

<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 50 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     1014 non-null   str    
 1   FASE_23                1014 non-null   str    
 2   INDE_2023              931 non-null    float64
 3   PEDRA_2023             931 non-null    str    
 4   TURMA_23               1014 non-null   str    
 5   NOME_ANONIMIZADO       1014 non-null   str    
 6   DATA_DE_NASC           1014 non-null   object 
 7   IDADE_2023             1014 non-null   object 
 8   GÊNERO                 1014 non-null   int64  
 9   ANO_INGRESSO           1014 non-null   int64  
 10  INSTITUICAO_ENSINO_23  1014 non-null   str    
 11  PEDRA 20               240 non-null    str    
 12  PEDRA 21               335 non-null    str    
 13  PEDRA 22               600 non-null    str    
 14  INDE_2022              600 non-null    float64
 15  CG_23          

### PEDE 2024

In [25]:
df_pede_2024_padronizado = padronizar_colunas_ano(df_pede_2024, 2024, ignorar_cols=['Ano_Ingresso', 'Data_de_Nasc', 'Gênero', 'INDE_2022', 'INDE_2023', 'Nome_Anonimizado', 'RA', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23'])
display(df_pede_2024_padronizado.head())

,RA,FASE_24,INDE_2024,PEDRA_24,TURMA_24,NOME_ANONIMIZADO,DATA_DE_NASC,IDADE_2024,GÊNERO,ANO_INGRESSO,INSTITUICAO_ENSINO_24,PEDRA 20,PEDRA 21,PEDRA 22,PEDRA 23,INDE_2022,INDE_2023,CG_24,CF_24,CT_24,QTDE_AVAL_2024,AVALIADOR1_24,REC_AV1_24,AVALIADOR2_24,REC_AV2_24,AVALIADOR3_24,AVALIADOR4_24,AVALIADOR5_24,AVALIADOR6_24,IAA_24,IEG_24,IPS_24,IPP_2024,REC_PSICOLOGIA_24,IDA_24,MATEMATICA_24,PORTUGUES_24,INGLES_24,INDICADO_24,ATINGIU_PV_24,IPV_24,IAN_24,FASE_IDEAL_24,DEFASAGEM_24,DESTAQUE_IEG_24,DESTAQUE_IDA_24,DESTAQUE_IPV_24,ESCOLA_24,ATIVO/_INATIVO_24,REC_AV3_24,REC_AV4_24,REC_AV5_24,REC_AV6_24
0,RA-1275,ALFA,7.611367,Ametista,ALFA A - G0/G1,Aluno-1275,2016-07-28,8,Masculino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,NaN,Avaliador-2,NaN,Avaliador-9,NaN,NaN,NaN,10.002,8.666667,6.26,5.625,NaN,8.0,10.0,6.0,NaN,NaN,NaN,5.446667,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Chácara Florida II,Cursando,NaN,NaN,NaN,NaN
1,RA-1276,ALFA,8.002867,Topázio,ALFA A - G0/G1,Aluno-1276,2016-10-16,8,Feminino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,NaN,Avaliador-2,NaN,Avaliador-9,NaN,NaN,NaN,10.002,9.333333,3.76,7.500,NaN,8.0,10.0,6.0,NaN,NaN,NaN,7.050000,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Chácara Florida II,Cursando,NaN,NaN,NaN,NaN
2,RA-1277,ALFA,7.9522,Ametista,ALFA A - G0/G1,Aluno-1277,2016-08-16,8,Masculino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,NaN,Avaliador-2,NaN,Avaliador-9,NaN,NaN,NaN,10.002,9.083333,3.76,7.500,NaN,8.0,10.0,6.0,NaN,NaN,NaN,7.046667,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Dom Pedro Villas Boas de Souza,Cursando,NaN,NaN,NaN,NaN
3,RA-868,ALFA,7.156367,Ametista,ALFA A - G0/G1,Aluno-868,2015-11-08,8,Masculino,2023,Pública,NaN,NaN,NaN,Topázio,NaN,8.63895,NaN,NaN,NaN,3,Avaliador-11,NaN,Avaliador-2,NaN,Avaliador-9,NaN,NaN,NaN,8.002,9.750000,3.76,6.875,NaN,7.0,8.0,6.0,NaN,NaN,NaN,7.213333,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN,EE Chácara Florida II,Cursando,NaN,NaN,NaN,NaN
4,RA-1278,ALFA,5.4442,Quartzo,ALFA A - G0/G1,Aluno-1278,2015-03-22,9,Masculino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,NaN,Avaliador-2,NaN,Avaliador-9,NaN,NaN,NaN,9.002,4.166667,3.76,5.000,NaN,7.5,8.0,7.0,NaN,NaN,NaN,4.173333,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN,EM Etelvina Delfim Simões,Cursando,NaN,NaN,NaN,NaN


In [26]:
mapa_genero = {'Feminino': 1, 'Masculino': 0}
df_pede_2024_padronizado['GÊNERO'] = df_pede_2024_padronizado['GÊNERO'].map(mapa_genero)

df_pede_2024_padronizado = obter_nova_fase_24(df_pede_2024_padronizado, 'FASE_24') # Obtendo coluna de fase corrigida para 2024

df_pede_2024_padronizado = obter_nova_turma_24(df_pede_2024_padronizado, 'TURMA_24') # Obtendo coluna de turma corrigida para 2024

# Obtendo a nova coluna de fase ideal padronizada
df_pede_2024_padronizado = obter_nova_fase(df_pede_2024_padronizado, 'FASE_IDEAL_24')

df_pede_2024_padronizado = criar_coluna_em_fase(df_pede_2024_padronizado, 'FASE_24', 'NOVA_FASE_IDEAL_24') 

df_pede_2024_padronizado = criar_coluna_veterano(df_pede_2024_padronizado, 'ANO_INGRESSO', 2024)

ajuste_pedra_24 = {
    'Agata': 'Ágata',
}
df_pede_2024_padronizado['PEDRA_24'] = df_pede_2024_padronizado['PEDRA_24'].replace(ajuste_pedra_24) 

ajuste_pedra_23 = {
    'Agata': 'Ágata',
}
df_pede_2024_padronizado['PEDRA 23'] = df_pede_2024_padronizado['PEDRA 23'].replace(ajuste_pedra_23) 

mapa_instituicao_ensino_24 = {
    'Pública': 'Pública',
    'Privada': 'Privada',
    'Concluiu o 3º EM': 'Concluiu o 3º EM',
    'Privada - Programa de Apadrinhamento': 'Privada',
    'Privada *Parcerias com Bolsa 100%': 'Privada',
    'Privada - Pagamento por *Empresa Parceira': 'Privada',
    'Privada - Programa de apadrinhamento': 'Privada',
    'Bolsista Universitário *Formado (a)': 'Universitário',
}
df_pede_2024_padronizado['INSTITUICAO_ENSINO_24'] = df_pede_2024_padronizado['INSTITUICAO_ENSINO_24'].map(mapa_instituicao_ensino_24) # Ajuste da coluna de instituição de ensino de 2024 para padronizar com 2023 e 2022

In [27]:
df_pede_2024_padronizado.info()

<class 'pandas.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 58 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   RA                     1156 non-null   str           
 1   FASE_24                1156 non-null   object        
 2   INDE_2024              1092 non-null   object        
 3   PEDRA_24               1092 non-null   str           
 4   TURMA_24               1156 non-null   object        
 5   NOME_ANONIMIZADO       1156 non-null   str           
 6   DATA_DE_NASC           1156 non-null   datetime64[us]
 7   IDADE_2024             1156 non-null   int64         
 8   GÊNERO                 1156 non-null   int64         
 9   ANO_INGRESSO           1156 non-null   int64         
 10  INSTITUICAO_ENSINO_24  1155 non-null   str           
 11  PEDRA 20               191 non-null    str           
 12  PEDRA 21               264 non-null    str           
 13  PEDRA 22      

# Criando dataframes

## Selecionando colunas

### 2022

In [28]:
df_2022 = df_pede_2022_padronizado[['RA', 'FASE_22','TURMA_22', 'DEFASAGEM_22', 'NOVA_FASE_IDEAL_22', 'IDADE_2022', 'GÊNERO', 'ANO_INGRESSO', 'INSTITUICAO_ENSINO_22','VETERANO', 'EM_FASE','QTDE_AVAL_2022','CG_22', 'CF_22', 'CT_22', 'IAA_22', 'IEG_22', 'IPS_22', 'IDA_22', 'IPV_22', 'IAN_22']]
analise_nulos(df_2022)

,nome_coluna,qtd_nulos,porcentagem_nulos
0,RA,0,0.0
1,FASE_22,0,0.0
2,TURMA_22,0,0.0
3,DEFASAGEM_22,0,0.0
4,NOVA_FASE_IDEAL_22,0,0.0
5,IDADE_2022,0,0.0
6,GÊNERO,0,0.0
7,ANO_INGRESSO,0,0.0
8,INSTITUICAO_ENSINO_22,0,0.0
9,VETERANO,0,0.0


### 2023

In [29]:
df_2023 = df_pede_2023_padronizado[['RA', 'FASE_23','TURMA_23', 'DEFASAGEM_23', 'NOVA_FASE_IDEAL_23', 'NOVA_IDADE_2023', 'GÊNERO', 'ANO_INGRESSO', 'INSTITUICAO_ENSINO_23','VETERANO', 'EM_FASE','QTDE_AVAL_2023','CG_23', 'CF_23', 'CT_23', 'IAA_23', 'IEG_23', 'IPS_23', 'IDA_23', 'IPV_23', 'IAN_23']]
analise_nulos(df_2023)

,nome_coluna,qtd_nulos,porcentagem_nulos
12,CG_23,1014,100.00
13,CF_23,1014,100.00
14,CT_23,1014,100.00
18,IDA_23,77,7.59
16,IEG_23,76,7.50
19,IPV_23,76,7.50
11,QTDE_AVAL_2023,76,7.50
17,IPS_23,69,6.80
15,IAA_23,63,6.21
0,RA,0,0.00


### 2024

In [30]:
df_2024 = df_pede_2024_padronizado[['RA', 'FASE_24','TURMA_24', 'DEFASAGEM_24', 'NOVA_FASE_IDEAL_24', 'IDADE_2024', 'GÊNERO', 'ANO_INGRESSO', 'INSTITUICAO_ENSINO_24','VETERANO', 'EM_FASE','QTDE_AVAL_2024','CG_24', 'CF_24', 'CT_24', 'IAA_24', 'IEG_24', 'IPS_24', 'IDA_24', 'IPV_24', 'IAN_24']]
analise_nulos(df_2024)

,nome_coluna,qtd_nulos,porcentagem_nulos
12,CG_24,1156,100.00
13,CF_24,1156,100.00
14,CT_24,1156,100.00
15,IAA_24,102,8.82
17,IPS_24,102,8.82
19,IPV_24,102,8.82
18,IDA_24,101,8.74
8,INSTITUICAO_ENSINO_24,1,0.09
4,NOVA_FASE_IDEAL_24,0,0.00
0,RA,0,0.00


## Dataframes para modelagem

### df_model_2022

In [31]:
df_total = pd.merge(df_2022, df_2023, on='RA', how='outer', suffixes=('_22', '_23'))

In [32]:
resultado = pd.merge(df_2022, df_2023, on='RA', how='outer', suffixes=('_22', '_23'), indicator=True)
resultado.head()

,RA,FASE_22,TURMA_22,DEFASAGEM_22,NOVA_FASE_IDEAL_22,IDADE_2022,GÊNERO_22,ANO_INGRESSO_22,INSTITUICAO_ENSINO_22,VETERANO_22,EM_FASE_22,QTDE_AVAL_2022,CG_22,CF_22,CT_22,IAA_22,IEG_22,IPS_22,IDA_22,IPV_22,IAN_22,FASE_23,TURMA_23,DEFASAGEM_23,NOVA_FASE_IDEAL_23,NOVA_IDADE_2023,GÊNERO_23,ANO_INGRESSO_23,INSTITUICAO_ENSINO_23,VETERANO_23,EM_FASE_23,QTDE_AVAL_2023,CG_23,CF_23,CT_23,IAA_23,IEG_23,IPS_23,IDA_23,IPV_23,IAN_23,_merge
0,RA-1,Fase 7,A,-1.0,Fase 8,19.0,1.0,2016.0,Pública,1.0,0.0,4.0,753.0,18.0,10.0,8.3,4.1,5.6,4.0,7.278,5.0,Fase 8,8E,0.0,Fase 8,20.0,1.0,2016.0,Privada,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,both
1,RA-10,Fase 7,A,-1.0,Fase 8,18.0,1.0,2021.0,Pública,1.0,0.0,4.0,752.0,17.0,9.0,8.3,5.2,5.0,4.1,7.056,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,RA-100,Fase 4,A,1.0,Fase 3,13.0,1.0,2019.0,Privada,1.0,0.0,4.0,268.0,22.0,7.0,8.8,7.8,5.0,7.6,7.250,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,RA-1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fase 0,ALFA U - G2/G3,0.0,Fase 0,8.0,1.0,2023.0,Pública,0.0,1.0,2.0,NaN,NaN,NaN,8.5,9.4,3.77,7.0,8.92,10.0,right_only
4,RA-1001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fase 0,ALFA U - G2/G3,-1.0,Fase 1,9.0,1.0,2023.0,Pública,0.0,0.0,2.0,NaN,NaN,NaN,9.0,9.1,7.52,7.8,9.17,5.0,right_only


In [33]:
map_evadiu = {
    'left_only': 1,
    'both': 0
}

df_resultado = resultado[resultado['_merge'].isin(['left_only','both'])].reset_index(drop=True)
df_resultado['EVADIU'] = df_resultado['_merge'].map(map_evadiu)
df_resultado

,RA,FASE_22,TURMA_22,DEFASAGEM_22,NOVA_FASE_IDEAL_22,IDADE_2022,GÊNERO_22,ANO_INGRESSO_22,INSTITUICAO_ENSINO_22,VETERANO_22,EM_FASE_22,QTDE_AVAL_2022,CG_22,CF_22,CT_22,IAA_22,IEG_22,IPS_22,IDA_22,IPV_22,IAN_22,FASE_23,TURMA_23,DEFASAGEM_23,NOVA_FASE_IDEAL_23,NOVA_IDADE_2023,GÊNERO_23,ANO_INGRESSO_23,INSTITUICAO_ENSINO_23,VETERANO_23,EM_FASE_23,QTDE_AVAL_2023,CG_23,CF_23,CT_23,IAA_23,IEG_23,IPS_23,IDA_23,IPV_23,IAN_23,_merge,EVADIU
0,RA-1,Fase 7,A,-1.0,Fase 8,19.0,1.0,2016.0,Pública,1.0,0.0,4.0,753.0,18.0,10.0,8.3,4.1,5.6,4.0,7.278,5.0,Fase 8,8E,0.0,Fase 8,20.0,1.0,2016.0,Privada,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,both,0.0
1,RA-10,Fase 7,A,-1.0,Fase 8,18.0,1.0,2021.0,Pública,1.0,0.0,4.0,752.0,17.0,9.0,8.3,5.2,5.0,4.1,7.056,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
2,RA-100,Fase 4,A,1.0,Fase 3,13.0,1.0,2019.0,Privada,1.0,0.0,4.0,268.0,22.0,7.0,8.8,7.8,5.0,7.6,7.250,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
3,RA-101,Fase 4,A,-1.0,Fase 5,15.0,1.0,2021.0,Pública,1.0,0.0,4.0,387.0,31.0,8.0,7.9,8.3,7.5,7.6,6.833,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
4,RA-102,Fase 4,A,0.0,Fase 4,14.0,0.0,2019.0,Pública,1.0,1.0,4.0,519.0,43.0,9.0,6.7,7.7,6.9,5.7,6.000,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855,RA-95,Fase 5,L,-2.0,Fase 7,17.0,1.0,2022.0,Pública,0.0,0.0,4.0,685.0,41.0,9.0,8.3,6.6,7.5,4.8,6.292,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
856,RA-96,Fase 5,L,-1.0,Fase 6,16.0,0.0,2021.0,Pública,1.0,0.0,4.0,295.0,19.0,5.0,9.2,7.8,6.9,6.7,8.500,5.0,Fase 6,6L,-1.0,Fase 7,17.0,0.0,2021.0,Privada,1.0,0.0,4.0,NaN,NaN,NaN,9.6,8.9,7.52,6.4,7.8400,5.0,both,0.0
857,RA-97,Fase 5,L,0.0,Fase 5,15.0,1.0,2017.0,Privada,1.0,1.0,4.0,328.0,22.0,6.0,7.5,8.8,5.0,6.9,6.292,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,1.0
858,RA-98,Fase 5,L,0.0,Fase 5,15.0,1.0,2017.0,Privada,1.0,1.0,4.0,2.0,2.0,2.0,8.8,10.0,9.4,8.5,9.667,10.0,Fase 7,7E,2.0,Fase 5,16.0,1.0,2017.0,Privada,1.0,0.0,NaN,NaN,NaN,NaN,7.9,NaN,2.52,NaN,NaN,10.0,both,0.0


In [34]:
colunas_finais_2022 = [col for col in df_resultado.columns if not col.endswith('_23') and not col.endswith('_2023')]
df_model_2022 = df_resultado[colunas_finais_2022].drop(columns=['_merge'], errors='ignore').copy()
analise_nulos(df_model_2022)

,nome_coluna,qtd_nulos,porcentagem_nulos
0,RA,0,0.0
1,FASE_22,0,0.0
2,TURMA_22,0,0.0
3,DEFASAGEM_22,0,0.0
4,NOVA_FASE_IDEAL_22,0,0.0
5,IDADE_2022,0,0.0
6,GÊNERO_22,0,0.0
7,ANO_INGRESSO_22,0,0.0
8,INSTITUICAO_ENSINO_22,0,0.0
9,VETERANO_22,0,0.0


In [ ]:
df_model_2022.to_csv('../app/data/processed/df_model_2022.csv', index=False, encoding='utf-8')

In [ ]:
df_resultado['EVADIU'].value_counts()

In [ ]:
analise_nulos(df_resultado)